In [0]:
-- Create dedicated audit schema for ETL pipeline monitoring
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.audit
COMMENT 'Audit schema for ETL pipeline execution tracking and diagnostics';

In [0]:
-- Enhanced pipeline audit log with execution metrics
CREATE OR REPLACE TABLE retail_lakehouse.audit.pipeline_execution_log
(
    RunID STRING COMMENT 'Unique identifier for pipeline run',
    PipelineStage STRING COMMENT 'Bronze/Silver/Gold/Incremental/SCD2/Validation/Archival',
    LayerName STRING COMMENT 'Specific layer being processed',
    TableName STRING COMMENT 'Target table name',
    Status STRING COMMENT 'SUCCESS/FAILED/RUNNING/SKIPPED',
    Severity STRING COMMENT 'INFO/WARNING/HIGH/CRITICAL',
    
    -- Performance metrics
    StartTimestamp TIMESTAMP COMMENT 'Execution start time',
    EndTimestamp TIMESTAMP COMMENT 'Execution end time',
    DurationSeconds BIGINT COMMENT 'Execution duration in seconds',
    
    -- Data metrics
    RowsRead BIGINT COMMENT 'Number of rows read from source',
    RowsWritten BIGINT COMMENT 'Number of rows written to target',
    RowsRejected BIGINT COMMENT 'Number of rows rejected due to quality issues',
    RowsUpdated BIGINT COMMENT 'Number of rows updated (for incremental)',
    RowsInserted BIGINT COMMENT 'Number of rows inserted (for incremental)',
    
    -- Error tracking
    ErrorMessage STRING COMMENT 'Detailed error message if failed',
    ErrorCode STRING COMMENT 'Error code for categorization',
    
    -- Audit metadata
    ExecutedBy STRING COMMENT 'User or job that executed this stage',
    ExecutionTimestamp TIMESTAMP COMMENT 'Timestamp when logged',
    NotebookPath STRING COMMENT 'Path to notebook that ran this stage'
)
USING DELTA
COMMENT 'Comprehensive audit log for tracking ETL pipeline execution and performance';

In [0]:
-- Data quality tracking table
CREATE OR REPLACE TABLE retail_lakehouse.audit.data_quality_log
(
    RunID STRING,
    TableName STRING,
    QualityCheckName STRING COMMENT 'Name of validation check',
    CheckType STRING COMMENT 'NULL_CHECK/DUPLICATE_CHECK/REFERENTIAL_INTEGRITY/BUSINESS_RULE',
    Status STRING COMMENT 'PASSED/FAILED',
    ExpectedValue STRING,
    ActualValue STRING,
    FailedRecordCount BIGINT,
    Severity STRING COMMENT 'INFO/WARNING/HIGH/CRITICAL',
    CheckTimestamp TIMESTAMP
)
USING DELTA
COMMENT 'Data quality validation results for pipeline monitoring';

In [0]:
-- Rejected records table for data quality issues
CREATE OR REPLACE TABLE retail_lakehouse.audit.rejected_records
(
    RunID STRING,
    SourceTable STRING,
    RejectionReason STRING,
    Severity STRING,
    RecordData STRING COMMENT 'JSON representation of rejected record',
    RejectionTimestamp TIMESTAMP
)
USING DELTA
COMMENT 'Tracks all records rejected during ETL processing';

-- Create rejected sales table with invalid data
CREATE OR REPLACE TABLE retail_lakehouse.audit.rejected_sales
USING DELTA
AS
SELECT
    *,
    'Invalid quantity (<= 0)' AS RejectionReason,
    'HIGH' AS Severity,
    CURRENT_TIMESTAMP AS RejectionTimestamp
FROM retail_lakehouse.bronze.sales
WHERE TRY_CAST(Quantity AS INT) IS NULL OR TRY_CAST(Quantity AS INT) <= 0;

In [0]:
-- File processing audit log
CREATE OR REPLACE TABLE retail_lakehouse.audit.file_processing_log
(
    RunID STRING,
    FileName STRING,
    FilePath STRING,
    ZoneName STRING COMMENT 'bronze/processed/archive',
    FileStatus STRING COMMENT 'ACTIVE/ARCHIVED/DELETED',
    FileSize BIGINT COMMENT 'File size in bytes',
    RecordCount BIGINT,
    ProcessedTimestamp TIMESTAMP,
    ArchivedTimestamp TIMESTAMP
)
USING DELTA
COMMENT 'Tracks file lifecycle in the pipeline';

In [0]:
-- PIPELINE HEALTH DASHBOARD
-- Real-time view of current pipeline execution status
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_pipeline_health_dashboard AS

WITH LatestRun AS (
    SELECT MAX(RunID) AS CurrentRunID
    FROM retail_lakehouse.audit.pipeline_execution_log
),

StageStatus AS (
    SELECT
        pel.PipelineStage,
        pel.Status,
        pel.StartTimestamp,
        pel.EndTimestamp,
        pel.DurationSeconds,
        SUM(pel.RowsWritten) AS TotalRowsProcessed,
        COUNT(DISTINCT pel.TableName) AS TablesAffected,
        MAX(pel.Severity) AS MaxSeverity
    FROM retail_lakehouse.audit.pipeline_execution_log pel
    INNER JOIN LatestRun lr ON pel.RunID = lr.CurrentRunID
    GROUP BY pel.PipelineStage, pel.Status, pel.StartTimestamp, pel.EndTimestamp, pel.DurationSeconds
)

SELECT
    PipelineStage,
    Status,
    TotalRowsProcessed,
    TablesAffected,
    DurationSeconds,
    CONCAT(ROUND(DurationSeconds / 60, 2), ' min') AS Duration,
    MaxSeverity,
    StartTimestamp,
    EndTimestamp,
    CASE
        WHEN Status = 'SUCCESS' THEN '✓ Completed'
        WHEN Status = 'FAILED' THEN '✗ Failed'
        WHEN Status = 'RUNNING' THEN '⟳ In Progress'
        ELSE Status
    END AS StatusIcon
FROM StageStatus
ORDER BY
    CASE PipelineStage
        WHEN 'Bronze' THEN 1
        WHEN 'Silver' THEN 2
        WHEN 'Gold' THEN 3
        WHEN 'Incremental' THEN 4
        WHEN 'SCD2' THEN 5
        WHEN 'Validation' THEN 6
        WHEN 'Archival' THEN 7
        ELSE 8
    END;

In [0]:
-- PIPELINE EXECUTION SUMMARY
-- Overall pipeline performance and success metrics
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_pipeline_execution_summary AS

SELECT
    COUNT(DISTINCT RunID) AS TotalPipelineRuns,
    COUNT(DISTINCT CASE WHEN Status = 'SUCCESS' THEN RunID END) AS SuccessfulRuns,
    COUNT(DISTINCT CASE WHEN Status = 'FAILED' THEN RunID END) AS FailedRuns,
    ROUND(COUNT(DISTINCT CASE WHEN Status = 'SUCCESS' THEN RunID END) * 100.0 / NULLIF(COUNT(DISTINCT RunID), 0), 2) AS SuccessRate,
    
    COUNT(*) AS TotalStageExecutions,
    SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) AS SuccessfulStages,
    SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailedStages,
    
    SUM(RowsRead) AS TotalRowsRead,
    SUM(RowsWritten) AS TotalRowsWritten,
    SUM(RowsRejected) AS TotalRowsRejected,
    
    SUM(CASE WHEN Severity = 'CRITICAL' THEN 1 ELSE 0 END) AS CriticalIssues,
    SUM(CASE WHEN Severity = 'HIGH' THEN 1 ELSE 0 END) AS HighSeverityIssues,
    SUM(CASE WHEN Severity = 'WARNING' THEN 1 ELSE 0 END) AS Warnings,
    
    AVG(DurationSeconds) AS AvgStageDurationSeconds,
    MAX(ExecutionTimestamp) AS LastExecutionTime,
    MIN(ExecutionTimestamp) AS FirstExecutionTime
    
FROM retail_lakehouse.audit.pipeline_execution_log;

In [0]:
-- LAYER-WISE PERFORMANCE BREAKDOWN
-- Detailed metrics for each pipeline layer
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_layer_performance AS

SELECT
    PipelineStage,
    COUNT(DISTINCT RunID) AS TotalRuns,
    
    -- Success metrics
    SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) AS SuccessCount,
    SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailureCount,
    ROUND(SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 2) AS SuccessRate,
    
    -- Data volume metrics
    SUM(RowsRead) AS TotalRowsRead,
    SUM(RowsWritten) AS TotalRowsWritten,
    SUM(RowsRejected) AS TotalRowsRejected,
    ROUND(SUM(RowsRejected) * 100.0 / NULLIF(SUM(RowsRead), 0), 2) AS RejectionRate,
    
    -- Performance metrics
    AVG(DurationSeconds) AS AvgDurationSeconds,
    MIN(DurationSeconds) AS MinDurationSeconds,
    MAX(DurationSeconds) AS MaxDurationSeconds,
    
    -- Throughput
    ROUND(SUM(RowsWritten) / NULLIF(SUM(DurationSeconds), 0), 2) AS AvgRowsPerSecond,
    
    MAX(ExecutionTimestamp) AS LastRun
    
FROM retail_lakehouse.audit.pipeline_execution_log
GROUP BY PipelineStage
ORDER BY
    CASE PipelineStage
        WHEN 'Bronze' THEN 1
        WHEN 'Silver' THEN 2
        WHEN 'Gold' THEN 3
        WHEN 'Incremental' THEN 4
        WHEN 'SCD2' THEN 5
        WHEN 'Validation' THEN 6
        WHEN 'Archival' THEN 7
        ELSE 8
    END;

In [0]:
-- FAILED EXECUTIONS DIAGNOSTIC VIEW
-- Detailed information about failed pipeline stages
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_failed_executions AS

SELECT
    RunID,
    PipelineStage,
    LayerName,
    TableName,
    Severity,
    ErrorCode,
    ErrorMessage,
    RowsRead,
    RowsWritten,
    RowsRejected,
    StartTimestamp,
    EndTimestamp,
    DurationSeconds,
    ExecutedBy,
    NotebookPath,
    ExecutionTimestamp
FROM retail_lakehouse.audit.pipeline_execution_log
WHERE Status = 'FAILED'
ORDER BY ExecutionTimestamp DESC, Severity DESC;

In [0]:
-- DATA QUALITY MONITORING
-- Summary of all data quality checks and validation results
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_data_quality_summary AS

SELECT
    TableName,
    CheckType,
    
    COUNT(*) AS TotalChecks,
    SUM(CASE WHEN Status = 'PASSED' THEN 1 ELSE 0 END) AS PassedChecks,
    SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailedChecks,
    ROUND(SUM(CASE WHEN Status = 'PASSED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS PassRate,
    
    SUM(FailedRecordCount) AS TotalFailedRecords,
    
    MAX(CASE WHEN Status = 'FAILED' THEN Severity END) AS HighestFailureSeverity,
    MAX(CheckTimestamp) AS LastCheckTime
    
FROM retail_lakehouse.audit.data_quality_log
GROUP BY TableName, CheckType
ORDER BY TableName, CheckType;

In [0]:
-- TABLE-WISE EXECUTION DETAILS
-- Granular view of each table's processing history
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_table_execution_details AS

WITH TableStats AS (
    SELECT
        LayerName,
        TableName,
        
        COUNT(DISTINCT RunID) AS TotalRuns,
        SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) AS SuccessCount,
        SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailureCount,
        
        SUM(RowsRead) AS TotalRowsRead,
        SUM(RowsWritten) AS TotalRowsWritten,
        SUM(RowsInserted) AS TotalInserted,
        SUM(RowsUpdated) AS TotalUpdated,
        SUM(RowsRejected) AS TotalRejected,
        
        AVG(DurationSeconds) AS AvgDurationSeconds,
        MAX(DurationSeconds) AS MaxDurationSeconds,
        MAX(ExecutionTimestamp) AS LastProcessed
        
    FROM retail_lakehouse.audit.pipeline_execution_log
    GROUP BY LayerName, TableName
),

LastStatus AS (
    SELECT DISTINCT
        LayerName,
        TableName,
        FIRST_VALUE(Status) OVER (PARTITION BY LayerName, TableName ORDER BY ExecutionTimestamp DESC) AS LastStatus
    FROM retail_lakehouse.audit.pipeline_execution_log
)

SELECT
    ts.LayerName,
    ts.TableName,
    ts.TotalRuns,
    ts.SuccessCount,
    ts.FailureCount,
    ts.TotalRowsRead,
    ts.TotalRowsWritten,
    ts.TotalInserted,
    ts.TotalUpdated,
    ts.TotalRejected,
    ts.AvgDurationSeconds,
    ts.MaxDurationSeconds,
    ts.LastProcessed,
    ls.LastStatus
FROM TableStats ts
LEFT JOIN LastStatus ls ON ts.LayerName = ls.LayerName AND ts.TableName = ls.TableName
ORDER BY ts.LayerName, ts.TableName;

In [0]:
-- PIPELINE EXECUTION TIMELINE
-- Chronological view of pipeline stages with duration
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_execution_timeline AS

SELECT
    RunID,
    PipelineStage,
    TableName,
    Status,
    
    StartTimestamp,
    EndTimestamp,
    DurationSeconds,
    
    -- Calculate time between stages
    LAG(EndTimestamp) OVER (PARTITION BY RunID ORDER BY StartTimestamp) AS PreviousStageEndTime,
    TIMESTAMPDIFF(SECOND, 
        LAG(EndTimestamp) OVER (PARTITION BY RunID ORDER BY StartTimestamp), 
        StartTimestamp
    ) AS GapFromPreviousStageSeconds,
    
    RowsWritten,
    Severity,
    ErrorMessage
    
FROM retail_lakehouse.audit.pipeline_execution_log
ORDER BY RunID DESC, StartTimestamp;

In [0]:
-- RECENT PIPELINE RUNS
-- Last 10 pipeline execution runs with overall status
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_recent_pipeline_runs AS

WITH RunSummary AS (
    SELECT
        RunID,
        MIN(StartTimestamp) AS PipelineStartTime,
        MAX(EndTimestamp) AS PipelineEndTime,
        SUM(DurationSeconds) AS TotalDurationSeconds,
        
        COUNT(*) AS TotalStages,
        SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) AS SuccessfulStages,
        SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailedStages,
        
        SUM(RowsRead) AS TotalRowsRead,
        SUM(RowsWritten) AS TotalRowsWritten,
        SUM(RowsRejected) AS TotalRowsRejected,
        
        MAX(CASE WHEN Severity IN ('HIGH', 'CRITICAL') THEN 1 ELSE 0 END) AS HasCriticalIssues,
        
        CASE
            WHEN SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) > 0 THEN 'FAILED'
            WHEN SUM(CASE WHEN Status = 'RUNNING' THEN 1 ELSE 0 END) > 0 THEN 'RUNNING'
            ELSE 'SUCCESS'
        END AS OverallStatus
        
    FROM retail_lakehouse.audit.pipeline_execution_log
    GROUP BY RunID
)

SELECT
    RunID,
    OverallStatus,
    PipelineStartTime,
    PipelineEndTime,
    CONCAT(ROUND(TotalDurationSeconds / 60, 2), ' min') AS TotalDuration,
    TotalStages,
    SuccessfulStages,
    FailedStages,
    TotalRowsWritten,
    TotalRowsRejected,
    CASE WHEN HasCriticalIssues = 1 THEN 'YES' ELSE 'NO' END AS CriticalIssues
FROM RunSummary
ORDER BY PipelineStartTime DESC
LIMIT 10;

In [0]:
-- FILE PROCESSING STATUS
-- Track which files have been processed and archived
CREATE OR REPLACE VIEW retail_lakehouse.audit.v_file_processing_status AS

SELECT
    ZoneName,
    FileName,
    FileStatus,
    RecordCount,
    ROUND(FileSize / 1024.0 / 1024.0, 2) AS FileSizeMB,
    ProcessedTimestamp,
    ArchivedTimestamp,
    CASE
        WHEN FileStatus = 'ACTIVE' THEN 'Currently Active'
        WHEN FileStatus = 'ARCHIVED' THEN CONCAT('Archived ', DATEDIFF(DAY, ArchivedTimestamp, CURRENT_TIMESTAMP), ' days ago')
        ELSE FileStatus
    END AS StatusDescription
FROM retail_lakehouse.audit.file_processing_log
ORDER BY ZoneName, ProcessedTimestamp DESC;

In [0]:
-- ==========================================
-- EXAMPLE: HOW TO LOG FROM YOUR NOTEBOOKS
-- ==========================================
-- Copy this pattern to each pipeline notebook

-- Sample logging for Bronze Layer
INSERT INTO retail_lakehouse.audit.pipeline_execution_log
VALUES (
    'RUN_001',                                    -- RunID (generate uuid() or use job_run_id)
    'Bronze',                                      -- PipelineStage
    'bronze',                                      -- LayerName
    'customers',                                   -- TableName
    'SUCCESS',                                     -- Status
    'INFO',                                        -- Severity
    CURRENT_TIMESTAMP - INTERVAL 5 MINUTES,       -- StartTimestamp
    CURRENT_TIMESTAMP,                            -- EndTimestamp
    300,                                          -- DurationSeconds
    1000,                                         -- RowsRead
    1000,                                         -- RowsWritten
    0,                                            -- RowsRejected
    0,                                            -- RowsUpdated
    1000,                                         -- RowsInserted
    NULL,                                         -- ErrorMessage
    NULL,                                         -- ErrorCode
    CURRENT_USER(),                               -- ExecutedBy
    CURRENT_TIMESTAMP,                            -- ExecutionTimestamp
    '/Users/.../02_bronze_layer'                  -- NotebookPath
);

-- Sample logging for failed execution
INSERT INTO retail_lakehouse.audit.pipeline_execution_log
VALUES (
    'RUN_001',
    'Gold',
    'gold',
    'fact_sales',
    'FAILED',
    'HIGH',
    CURRENT_TIMESTAMP - INTERVAL 2 MINUTES,
    CURRENT_TIMESTAMP,
    120,
    5000,
    0,
    100,
    0,
    0,
    'Foreign key constraint violation: customer_id not found in dim_customer',
    'REF_INTEGRITY_ERROR',
    CURRENT_USER(),
    CURRENT_TIMESTAMP,
    '/Users/.../04_gold_layer'
);

In [0]:
-- View the current pipeline health status
SELECT * FROM retail_lakehouse.audit.v_pipeline_health_dashboard;

In [0]:
-- View overall pipeline execution metrics
SELECT * FROM retail_lakehouse.audit.v_pipeline_execution_summary;

In [0]:
-- View performance metrics by pipeline stage
SELECT * FROM retail_lakehouse.audit.v_layer_performance;

In [0]:
-- View all failed pipeline stages with error details
SELECT * FROM retail_lakehouse.audit.v_failed_executions;

In [0]:
-- View the last 10 pipeline runs with overall status
SELECT * FROM retail_lakehouse.audit.v_recent_pipeline_runs;

In [0]:
-- View data quality validation results by table
SELECT * FROM retail_lakehouse.audit.v_data_quality_summary;